In [1]:
using ComputableDAGs
using Pkg
Pkg.develop(; path="/home/reinha57/repos/QEDFeynman.jl/")
using QEDFeynman
using RuntimeGeneratedFunctions
using BenchmarkTools
using QEDcore, QEDprocesses
using Logging
using JLD2
using CUDA

RuntimeGeneratedFunctions.init(@__MODULE__)

MODEL = PerturbativeQED()

   Resolving package versions...
  No Changes to `~/repos/ComputableDAGs.jl/Project.toml`
  No Changes to `~/repos/ComputableDAGs.jl/Manifest.toml`


perturbative QED

In [ ]:
N = 56 * 256
INSTANCE_STR = "kkkke->ke"
INSTANCE = parse_process(INSTANCE_STR, QEDModel())
g = graph(INSTANCE)

cu_inputs = CuVector([
    PhaseSpacePoint(
        INSTANCE,
        MODEL,
        PhasespaceDefinition(SphericalCoordinateSystem(), ElectronRestFrame()),
        tuple((rand(SFourMomentum) for _ in 1:number_incoming_particles(INSTANCE))...),
        tuple((rand(SFourMomentum) for _ in 1:number_outgoing_particles(INSTANCE))...),
    ) for _ in 1:N
])
cu_outputs = CuVector([0.0 for _ in 1:N])
k_unopt = eval(kernel(CUDAGPU, g, INSTANCE, @__MODULE__))

compute__048ae296_eada_11ef_26cb_7fd5db876979 (generic function with 1 method)

In [3]:
optimize_to_fixpoint!(ReductionOptimizer(), g)
k_opt = eval(kernel(CUDAGPU, g, INSTANCE, @__MODULE__))

compute__0ad50992_eada_11ef_3546_c9d23bca8865 (generic function with 1 method)

In [4]:
K = @cuda launch = false k_unopt(cu_inputs, cu_outputs, N)
K_opt = @cuda launch = false k_opt(cu_inputs, cu_outputs, N)

CUDA.HostKernel for compute__0ad50992_eada_11ef_3546_c9d23bca8865(CuDeviceVector{PhaseSpacePoint{ScatteringProcess{Tuple{Photon, Photon, Photon, Photon, Photon, Electron}, Tuple{Photon, Electron}, Tuple{PolarizationX, PolarizationX, PolarizationX, PolarizationX, PolarizationX, SpinUp}, Tuple{PolarizationX, SpinUp}}, PerturbativeQED, PhasespaceDefinition{SphericalCoordinateSystem, ElectronRestFrame}, Tuple{ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Electron, SFourMomentum}}, Tuple{ParticleStateful{Outgoing, Photon, SFourMomentum}, ParticleStateful{Outgoing, Electron, SFourMomentum}}, SFourMomentum}, 1}, CuDeviceVector{Float64, 1}, Int64)

In [5]:
K_inlined = @cuda launch = false always_inline = true k_unopt(cu_inputs, cu_outputs, N)

CUDA.HostKernel for compute__048ae296_eada_11ef_26cb_7fd5db876979(CuDeviceVector{PhaseSpacePoint{ScatteringProcess{Tuple{Photon, Photon, Photon, Photon, Photon, Electron}, Tuple{Photon, Electron}, Tuple{PolarizationX, PolarizationX, PolarizationX, PolarizationX, PolarizationX, SpinUp}, Tuple{PolarizationX, SpinUp}}, PerturbativeQED, PhasespaceDefinition{SphericalCoordinateSystem, ElectronRestFrame}, Tuple{ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Electron, SFourMomentum}}, Tuple{ParticleStateful{Outgoing, Photon, SFourMomentum}, ParticleStateful{Outgoing, Electron, SFourMomentum}}, SFourMomentum}, 1}, CuDeviceVector{Float64, 1}, Int64)

In [6]:
@info CUDA.memory(K)
@device_code_ptx io = open("unopt.ptx", write=true) @cuda launch = false k_unopt(cu_inputs, cu_outputs, N)
CUDA.@device_code dir = "./devcode_unopt" @cuda launch = false k_unopt(cu_inputs, cu_outputs, N)
@info CUDA.memory(K_opt)
@device_code_ptx io = open("opt.ptx", write=true) @cuda launch = false k_opt(cu_inputs, cu_outputs, N)
CUDA.@device_code dir = "./devcode_opt" @cuda launch = false k_opt(cu_inputs, cu_outputs, N)
@info CUDA.memory(K_inlined)
@device_code_ptx io = open("inlined.ptx", write=true) @cuda launch = false always_inline = true k_opt(cu_inputs, cu_outputs, N)

@info CUDA.registers(K)
@info CUDA.registers(K_opt)
@info CUDA.registers(K_inlined)

┌ Info: (local = 1736216, shared = 0, constant = 0)
└ @ Main /home/reinha57/repos/ComputableDAGs.jl/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W5sdnNjb2RlLXJlbW90ZQ==.jl:1
┌ Info: (local = 403528, shared = 0, constant = 0)
└ @ Main /home/reinha57/repos/ComputableDAGs.jl/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W5sdnNjb2RlLXJlbW90ZQ==.jl:4
┌ Info: (local = 34608, shared = 0, constant = 0)
└ @ Main /home/reinha57/repos/ComputableDAGs.jl/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W5sdnNjb2RlLXJlbW90ZQ==.jl:7
┌ Info: 255
└ @ Main /home/reinha57/repos/ComputableDAGs.jl/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W5sdnNjb2RlLXJlbW90ZQ==.jl:10
┌ Info: 255
└ @ Main /home/reinha57/repos/ComputableDAGs.jl/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W5sdnNjb2RlLXJlbW90ZQ==.jl:11
┌ Info: 255
└ @ Main /home/reinha57/repos/ComputableDAGs.jl/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W5sdnNjb2RlLXJlb

In [ ]:
CUDA.@sync (@cuda threads=32 blocks=N÷32 k_opt(cu_inputs, cu_outputs, N))

OutOfGPUMemoryError: Out of GPU memory
Effective GPU memory usage: 0.41% (99.000 MiB/23.599 GiB)
Memory pool usage: 8.250 KiB (32.000 MiB reserved)


In [8]:
CUDA.@sync (@cuda threads = 32 blocks = N ÷ 32 k_unopt(cu_inputs, cu_outputs, N))

CuError: CUDA error: invalid argument (code 1, ERROR_INVALID_VALUE)

In [9]:
g

Graph:
  Nodes: Total: 2234, ComputeTaskQED_U: 8, DataTask: 1121, 
         ComputeTaskQED_V: 312, ComputeTaskQED_S1: 72, ComputeTaskQED_S2: 720, 
         ComputeTaskQED_Sum: 1
  Edges: 3977
  Total Compute Effort: 684551.0
  Total Data Transfer: 274944.0
  Total Compute Intensity: 2.4897833740689013


In [10]:
using StaticArrays
using CUDA

N = 256
in = CuVector([rand(Float64) for _ in 1:N])
out = similar(in)

function mwe(input_vector, output_vector, n::Int64)
    id = (blockIdx().x - 1) * blockDim().x + threadIdx().x
    if (id > n)  
        return
    end
    @inbounds x = input_vector[id]
    y = MArray{Tuple{50},Float64}(undef)
    Base.Cartesian.@nexprs 50 i -> begin
        z = MArray{Tuple{50},Float64}(undef)
        Base.Cartesian.@nexprs 50 j -> begin
            @inbounds z[j] = tan(x + i) * tan(x * j)
        end
        @inbounds y[i] = sum(z)
    end
    @inbounds output_vector[id] = sum(y)
    return nothing
end

@info CUDA.memory(@cuda launch = false mwe(in, out, N))
@device_code_ptx io = open("mwe.ptx", write=true) @cuda launch = false mwe(in, out, N)

CUDA.@sync (@cuda threads=32 blocks=N÷32 mwe(in, out, N))
out

┌ Info: (local = 40040, shared = 0, constant = 0)
└ @ Main /home/reinha57/repos/ComputableDAGs.jl/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X12sdnNjb2RlLXJlbW90ZQ==.jl:26


256-element CuArray{Float64, 1, CUDA.DeviceMemory}:
  -5502.058740956282
    927.6942811955494
 505374.68419754156
  34404.23540218507
  -2204.6416765007534
 -16237.94394618301
 -30174.964261610035
   4975.66694508631
   1976.7283563724507
  -1119.2745149419948
      ⋮
    372.6021712298688
  -3154.543440366063
   1086.7561905657674
  -1377.5782090605755
   -629.3129709600618
     65.12593070054744
    264.11371646966893
   7830.231257414433
    260.5380578182506

In [ ]:
CUDA.@profile (CUDA.@sync (@cuda threads=32 blocks=N÷32 k_opt(cu_inputs, cu_outputs, N)))
CUDA.@profile (CUDA.@sync (@cuda threads=32 blocks=N÷32 k_unopt(cu_inputs, cu_outputs, N)))